# <h1 style='background:#f0c2c1; border:2; border-radius: 10px; font-size:300%; font-weight: bold; color:black'><center>Detecting Disaster from Tweets</center></h1> 
 


# Introduction 📝

Twitter has become an important communication channel in times of emergency.
The ubiquitousness of smartphones enables people to announce an emergency they’re observing in real-time. Because of this, more agencies are interested in programatically monitoring Twitter

🎯 **Goal:** The objective is to build a machine learning model that predicts which Tweets are about real disasters and which one’s aren’t. 


**🧪 Evaluation metric:** F1 Score
The F1 score is a commonly used metric for evaluating the performance of a binary classification model. It is defined as the harmonic mean of precision and recall, and can be written as:

$$F1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

where precision is the number of true positive predictions divided by the total number of positive predictions, and recall is the number of true positive predictions divided by the total number of actual positives.

In this formula, the precision and recall values can be represented as:

$$\text{precision} = \frac{TP}{TP + FP}$$

$$\text{recall} = \frac{TP}{TP + FN}$$

where $TP$ represents the number of true positive predictions, $FP$ represents the number of false positive predictions, and $FN$ represents the number of false negative predictions.

Therefore, the F1 score can be rewritten as:

$$F1 = 2 \cdot \frac{\frac{TP}{TP + FP} \cdot \frac{TP}{TP + FN}}{\frac{TP}{TP + FP} + \frac{TP}{TP + FN}}$$






In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

import nltk 
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize 
from nltk.stem import SnowballStemmer

from sklearn import model_selection, metrics, preprocessing, ensemble, model_selection, metrics
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Conv1D, Bidirectional, LSTM, Dense, Dropout, Input, SpatialDropout1D
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam


# 📚 Data Preprocessing

In [ ]:
# Rreading train dataset
file_path = "/kaggle/input/nlp-getting-started/train.csv"
raw_data = pd.read_csv(file_path)
print("Data points count: ", raw_data['id'].count())
raw_data.head()


**Data Description**
* id - a unique identifier for each tweet
* text - the text of the tweet
* location - the location the tweet was sent from (may be blank)
* keyword - a particular keyword from the tweet (may be blank)
* target - in train.csv only, this denotes whether a tweet is about a real disaster (1) or not (0)

In [ ]:
# Plotting target value counts
plt.figure(figsize=(10,8))
ax = raw_data['target'].value_counts().sort_values().plot(kind="bar")
ax.grid(axis="y")
plt.suptitle("Target Value Counts", fontsize=20)
plt.show()


As you can see there are more data points with the label 0 meaning tweets that are not disaster tweets and fewer data points with the label 1 which is tweets that are related to a disaster. Usually, for data that has some skewed labels, it is recommended to use an F-score instead of accuracy for model evaluation.

### 📚Missing Data

In [ ]:
print("Number of missing data for column keyword: ", raw_data['keyword'].isna().sum())
print("Number of missing data for column location: ", raw_data['location'].isna().sum())
print("Number of missing data for column text: ", raw_data['text'].isna().sum())
print("Number of missing data for column target: ", raw_data['target'].isna().sum())


In [ ]:
plt.figure(figsize=(15,8))
sns.heatmap(raw_data.drop('id', axis=1).isnull(), cbar=False, cmap="GnBu").set_title("Missing data for each column")
plt.show()

 The heatmap above shows that the column “keyword” has very few missing data points and I will fill the missing data points and use this column as a feature. Column “location” has very missing data and the quality of data is very bad. It has values that are not related to location. So I decided not to use this column and drop it. Column “text” which is the main column that has the actual tweet needs to be processed and get cleaned. It has no missing data.

### 📚 Cleaning Data

I have also noticed that there are some tweets that contain less than 3 words and I think two words sentences might not be able to transfer the knowledge very well. So to get a sense of the number of words sentences are made of lets look at the histogram of word count per sentence.

In [ ]:
plt.figure(figsize=(15,8))
raw_data['word_count'] = raw_data['text'].apply(lambda x: len(x.split(" ")) )
sns.distplot(raw_data['word_count'].values, hist=True, kde=True, kde_kws={"shade": True})
plt.axvline(raw_data['word_count'].describe()['25%'], ls="--")
plt.axvline(raw_data['word_count'].describe()['50%'], ls="--")
plt.axvline(raw_data['word_count'].describe()['75%'], ls="--")

plt.grid()
plt.suptitle("Word count histogram")
plt.show()

# remove rows with under 3 words
raw_data = raw_data[raw_data['word_count']>2]
raw_data = raw_data.reset_index()


As we can see the majority of tweets are between 11 to 19 words, so I have decided to remove tweets with less than 2 words. I believe sentences with 3 words can say enough about the tweet. It might be a good idea to also remove tweets with more than 25–30 words as they might slow down the training time.

In [ ]:
print("25th percentile: ", raw_data['word_count'].describe()['25%'])
print("mean: ", raw_data['word_count'].describe()['50%'])
print("75th percentile: ", raw_data['word_count'].describe()['75%'])

# Data Cleaning and Preprocessing:

Common steps for data cleaning on the NLP task dealing with tweets are removing special characters, removing stop words, removing URLs, removing numbers, and doing word stemming. But let’s first get more familiar with some NLP data preprocessing concepts:

**Vectorization:**

Word vectorization is a technique of mapping words to real numbers, or better say vector of real numbers. I have used vectorization from both Sklearn and Keras libraries.

**Tokenization:**

Tokenization is the task of breaking a phrase which can be anything such as a sentence, a paragraph, or just a text into smaller sections such as a series of words, a series of characters, or a series of subwords, and they are called tokens. One use of tokenization is to generate tokens from a text and later convert tokens to numbers (vectorization).

**Padding:**

Neural network models require to have inputs to have the same shape and same size, meaning all tweets that are inputted into the model one by one has to have exact same length, that is what padding is useful here. Every tweet in the dataset has a different number of words, we will set a maximum number of words for each tweet, if a tweet is longer then we can drop some words if the tweet has fewer words than the max we can fill either beginning or end of the tweet with fix value such as ‘0’.

**Stemming:**

The task of stemming is to reduce extra characters from a word to its root or base of a word. For example, stemming both words “working” and “worked” becomes “work”.

I used Snowball Stemmer which is a stemming algorithm (also known as the Porter2 stemming algorithm). It is a better version of the Porter Stemmer since some issues were fixed in this stemmer.

**Word Embedding:**

A word embedding is a learned representation for text where words that have the same meaning have a similar representation. Each word is mapped to one vector and the vector values are learned in a way that resembles a neural network.

In [ ]:
# Clean text columns
stop_words = set(stopwords.words('english'))
stemmer = SnowballStemmer('english')


def clean_text(each_text):

    # remove URL from text
    each_text_no_url = re.sub(r"http\S+", "", each_text)
    
    # remove numbers from text
    text_no_num = re.sub(r'\d+', '', each_text_no_url)

    # tokenize each text
    word_tokens = word_tokenize(text_no_num)
    
    # remove sptial character
    clean_text = []
    for word in word_tokens:
        clean_text.append("".join([e for e in word if e.isalnum()]))

    # remove stop words and lower
    text_with_no_stop_word = [w.lower() for w in clean_text if not w in stop_words]  

    # do stemming
    stemmed_text = [stemmer.stem(w) for w in text_with_no_stop_word]
    
    return " ".join(" ".join(stemmed_text).split())


raw_data['clean_text'] = raw_data['text'].apply(lambda x: clean_text(x) )
raw_data['keyword'] = raw_data['keyword'].fillna("none")
raw_data['clean_keyword'] = raw_data['keyword'].apply(lambda x: clean_text(x) )


To be able to use both “text” and “keyword” columns there are various approaches to apply but one simple approach that I have applied was to combine both of these two features into a new feature called “keyword_text”



In [ ]:
# Combine column 'clean_keyword' and 'clean_text' into one
raw_data['keyword_text'] = raw_data['clean_keyword'] + " " + raw_data["clean_text"]


In [ ]:
raw_data["clean_text"]

## Prepare train and test data

In [ ]:
feature = 'keyword_text'
label = "target"

# split train and test
X_train, X_test,y_train, y_test = model_selection.train_test_split(raw_data[feature],
                                                                   raw_data[label],
                                                                   test_size=0.3,
                                                                   random_state=0, 
                                                                   shuffle=True)


# Gradient Boosting Classifier

In [ ]:
X_train_GBC = X_train.values.reshape(-1)
x_test_GBC = X_test.values.reshape(-1)


As already mentioned about vectorization we have to convert text to numbers as machine learning models can only work with a number so we use “Countervectorize” here. We do fit and transform on train data and only transform on test data. Make sure there is no fitting happens on test data.


In [ ]:
# Vectorize text
vectorizer = CountVectorizer()
X_train_GBC = vectorizer.fit_transform(X_train_GBC)
x_test_GBC = vectorizer.transform(x_test_GBC)


In [ ]:
# Train the model
model = ensemble.GradientBoostingClassifier(learning_rate=0.1,                                            
                                            n_estimators=2000,
                                            max_depth=9,
                                            min_samples_split=6,
                                            min_samples_leaf=2,
                                            max_features=8,
                                            subsample=0.9)
model.fit(X_train_GBC, y_train)


In [ ]:
# Evaluate the model
predicted_prob = model.predict_proba(x_test_GBC)[:,1]
predicted = model.predict(x_test_GBC)

accuracy = metrics.accuracy_score(predicted, y_test)
print("Test accuracy: ", accuracy)
print(metrics.classification_report(y_test, predicted, target_names=["0", "1"]))
print("Test F-scoare: ", metrics.f1_score(y_test, predicted))

In [ ]:
# Plot confusion matrix
conf_matrix = metrics.confusion_matrix(y_test, predicted)

fig, ax = plt.subplots()
sns.heatmap(conf_matrix, cbar=False, cmap='Reds', annot=True, fmt='d')
ax.set(xlabel="Predicted Value", ylabel="True Value", title="Confusion Matrix")
ax.set_yticklabels(labels=['0', '1'], rotation=0)

plt.show()

# LSTM

#### GloVe embedding 

GloVe (Global Vectors for Word Representation) is a word embedding technique that aims to create word vectors that capture the semantic and syntactic meaning of words. It is based on the co-occurrence matrix of words in a corpus of text, which captures how often pairs of words appear together in the corpus.

The basic idea behind GloVe is to factorize the co-occurrence matrix into a product of two matrices, one for the words and one for the contexts (the words that appear in the same context as the target word). The resulting word and context vectors are then combined to create a single vector for each word that captures both its semantic and syntactic information.

To do this, GloVe defines a weighted least squares regression problem, where the goal is to learn a set of word vectors such that their dot product is equal to the logarithm of the words' co-occurrence probability. The objective function is:

J = ∑i,j f(Xij) (wi . wj + bwi + bwj - log(Xij))^2

where i and j are word indices, Xij is the number of times word i and word j co-occur in a window of a certain size, and f is a weighting function that reduces the influence of very common word pairs. The variables wi and wj are the word vectors, and bwi and bwj are bias terms.

The objective function is optimized using gradient descent to learn the word vectors that minimize the error. Once the optimization is complete, each word is represented as a vector of a fixed length, typically 100 to 300 dimensions, which can be used in various natural language processing tasks such as text classification, sentiment analysis, and machine translation.

Overall, GloVe embedding works by learning a low-dimensional vector representation for each word based on the co-occurrence information of the words in the text corpus. The resulting word vectors capture the semantic and syntactic information of the words and can be used as input to various NLP tasks.





In [ ]:
# Define some hyperparameters
path_to_glove_file = '/kaggle/input/glove6b/glove.6B.300d.txt'
embedding_dim = 300
learning_rate = 1e-3
batch_size = 1024
epochs = 20
sequence_len = 100


In [ ]:
# Define train and test labels
y_train_LSTM = y_train.values.reshape(-1,1)
y_test_LSTM = y_test.values.reshape(-1,1)

print("Training Y shape:", y_train_LSTM.shape)
print("Testing Y shape:", y_test_LSTM.shape)


In [ ]:
# Tokenize train data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
print("Vocabulary Size: ", vocab_size)


In [ ]:
# Pad train and test 
X_train = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=sequence_len)
X_test = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=sequence_len)

print("Training X shape: ", X_train.shape)
print("Testing X shape: ", X_test.shape)


In [ ]:
# Read word embeddings
embeddings_index = {}
with open(path_to_glove_file) as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs

print("Found %s word vectors." % len(embeddings_index))


In [ ]:
# Define embedding layer in Keras
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
        
embedding_layer = tf.keras.layers.Embedding(vocab_size,
                                            embedding_dim,
                                            weights=[embedding_matrix],
                                            input_length=sequence_len,
                                            trainable=False)


For the LSTM model,started with an embedding layer to generate an embedding vector for each input sequence. Then used a convolution model to lower the number of features followed by a bidirectional LSTM layer. The last layer is a dense layer. Because it is a binary classification we use sigmoid as an activation function.

In [ ]:
# Define model architecture
sequence_input = Input(shape=(sequence_len, ), dtype='int32')
embedding_sequences = embedding_layer(sequence_input)

x = Conv1D(128, 5, activation='relu')(embedding_sequences)
x = Bidirectional(LSTM(128, dropout=0.5, recurrent_dropout=0.2))(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)
model = Model(sequence_input, outputs)
model.summary()


In [ ]:
# Optimize the model
model.compile(optimizer=Adam(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
# Train the LSTM Model
history = model.fit(X_train,
                    y_train,
                    batch_size=batch_size,
                    epochs=epochs, 
                    validation_data=(X_test, y_test))


In [ ]:
# Evaluate the model
predicted = model.predict(X_test, verbose=1, batch_size=10000)

y_predicted = [1 if each > 0.5 else 0 for each in predicted]

score, test_accuracy = model.evaluate(X_test, y_test, batch_size=10000)

print("Test Accuracy: ", test_accuracy)
print(metrics.classification_report(list(y_test), y_predicted))


In [ ]:
# Plot confusion matrix
conf_matrix = metrics.confusion_matrix(y_test, y_predicted)

fig, ax = plt.subplots()
sns.heatmap(conf_matrix, cbar=False, cmap='Reds', annot=True, fmt='d')
ax.set(xlabel="Predicted Value", ylabel="True Value", title="Confusion Matrix")
ax.set_yticklabels(labels=['0', '1'], rotation=0)
plt.show()


<center><div class="alert alert-block alert-alert" style="margin: 2em; line-height: 1.7em; font-family: Verdana;">
    <b style="font-size: 18px;">👏 &nbsp; Thank You&nbsp; 👏</b><br><br><b style="font-size: 10px; color: darkgrey">If you find useful then please click "upvote" button in the top-right -- 
    it's very encouraging to notebook authors to know when people appreciate the work.</b> 😅
</div></center>